In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


In [2]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")

CUDA available: True
GPU device: NVIDIA A40
Number of GPUs: 1


In [3]:
# Explore the repository structure
repo_path = '/net/scratch2/smallyan/function_vectors_eval'

for root, dirs, files in os.walk(repo_path):
    level = root.replace(repo_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')

function_vectors_eval/
  .gitignore
  fv_overview.png
  documentation.pdf
  plan.md
  CodeWalkthrough.md
  fv_environment.yml
  src/
    portability_eval.py
    test_numheads.py
    compute_indirect_effect.py
    vocab_reconstruction.py
    __init__.py
    compute_avg_hidden_state.py
    natural_text_eval.py
    evaluate_function_vector.py
    compute_average_activations.py
    __pycache__/
      evaluate_function_vector.cpython-311.pyc
      __init__.cpython-311.pyc
      compute_indirect_effect.cpython-311.pyc
    utils/
      eval_utils.py
      prompt_utils.py
      intervention_utils.py
      extract_utils.py
      __init__.py
      model_utils.py
      __pycache__/
        model_utils.cpython-311.pyc
        intervention_utils.cpython-311.pyc
        __init__.cpython-311.pyc
        prompt_utils.cpython-311.pyc
        extract_utils.cpython-311.pyc
        eval_utils.cpython-311.pyc
    eval_scripts/
      eval_fv.sh
      eval_numheads.sh
      eval_template_portability.sh
     

In [4]:
# Read the Plan file
with open(f'{repo_path}/plan.md', 'r') as f:
    plan_content = f.read()
print(plan_content)

# Plan
## Objective
To investigate whether autoregressive transformer language models develop compact vector representations of input-output functions (called function vectors) within their hidden states during in-context learning, and to characterize these representations across diverse tasks and models.

## Hypothesis
1. A small number of attention heads transport a compact representation of the demonstrated task (function vector) that is robust to changes in context and can trigger task execution in zero-shot and natural text settings.
2. Function vectors contain information encoding the output space of the function, but this information alone is not sufficient to reconstruct a working function vector.
3. Function vectors can be composed through vector algebra to create vectors that trigger new complex tasks combining constituent tasks.

## Methodology
1. Apply causal mediation analysis to identify attention heads with highest average indirect effect (AIE) across diverse ICL tasks. 

In [5]:
# Read the CodeWalkthrough.md file
with open(f'{repo_path}/CodeWalkthrough.md', 'r') as f:
    walkthrough_content = f.read()
print(walkthrough_content)

# Function Vectors in Large Language Models
### [Project Website](https://functions.baulab.info) | [Arxiv Preprint](https://arxiv.org/abs/2310.15213) | [OpenReview](https://openreview.net/forum?id=AwyxtyMwaG)

This repository contains data and code for the paper: [Function Vectors in Large Language Models](https://arxiv.org/pdf/2310.15213).

<p align="left">
<img src="https://functions.baulab.info/images/Paper/fv-demonstrations.png" style="width:100%;"/>
</p> 

## Setup

We recommend using conda as a package manager. 
The environment used for this project can be found in the `fv_environment.yml` file.
To install, you can run: 
```
conda env create -f fv_environment.yml
conda activate fv
```

## Demo Notebook
Checkout `notebooks/fv_demo.ipynb` for a jupyter notebook with a demo of how to create a function vector and use it in different contexts.

## Data
The datasets used in our project can be found in the `dataset_files` folder.

## Code
Our main evaluation scripts are contained in the

In [6]:
# Read the implementation notebook
import json

with open(f'{repo_path}/notebooks/fv_demo.ipynb', 'r') as f:
    notebook_content = json.load(f)

# Print out the cells content
for i, cell in enumerate(notebook_content['cells']):
    cell_type = cell['cell_type']
    source = ''.join(cell['source'])
    print(f"=== Cell {i} ({cell_type}) ===")
    print(source[:1500] if len(source) > 1500 else source)
    print("\n")

=== Cell 0 (code) ===
%load_ext autoreload
%autoreload 2


=== Cell 1 (code) ===
import os, re, json
import torch, numpy as np

import sys
sys.path.append('..')
torch.set_grad_enabled(False)

from src.utils.extract_utils import get_mean_head_activations, compute_universal_function_vector
from src.utils.intervention_utils import fv_intervention_natural_text, function_vector_intervention
from src.utils.model_utils import load_gpt_model_and_tokenizer
from src.utils.prompt_utils import load_dataset, word_pairs_to_prompt_data, create_prompt
from src.utils.eval_utils import decode_to_vocab, sentence_eval


=== Cell 2 (markdown) ===
## Load model & tokenizer


=== Cell 3 (code) ===
model_name = 'EleutherAI/gpt-j-6b'
model, tokenizer, model_config = load_gpt_model_and_tokenizer(model_name)
EDIT_LAYER = 9


=== Cell 4 (markdown) ===
## Load dataset and Compute task-conditioned mean activations


=== Cell 5 (code) ===
dataset = load_dataset('antonym', seed=0)
mean_activations = get_mean_head_act

In [7]:
# Check if there are any outputs in the notebook cells
for i, cell in enumerate(notebook_content['cells']):
    if cell['cell_type'] == 'code' and 'outputs' in cell and cell['outputs']:
        print(f"=== Cell {i} Outputs ===")
        for output in cell['outputs']:
            if 'text' in output:
                print(''.join(output['text']))
            elif 'data' in output and 'text/plain' in output['data']:
                print(''.join(output['data']['text/plain']))
        print("\n")

In [8]:
# Check if there are any outputs
for i, cell in enumerate(notebook_content['cells']):
    if cell['cell_type'] == 'code':
        outputs = cell.get('outputs', [])
        if outputs:
            print(f"=== Cell {i} has {len(outputs)} output(s) ===")
            for j, output in enumerate(outputs):
                print(f"  Output {j}: type={output.get('output_type', 'unknown')}")
                if 'text' in output:
                    text = ''.join(output['text']) if isinstance(output['text'], list) else output['text']
                    print(f"    Text: {text[:500]}")
        else:
            print(f"=== Cell {i}: No outputs ===")

=== Cell 0: No outputs ===
=== Cell 1: No outputs ===
=== Cell 3: No outputs ===
=== Cell 5: No outputs ===
=== Cell 7: No outputs ===
=== Cell 9: No outputs ===
=== Cell 12: No outputs ===
=== Cell 14: No outputs ===
=== Cell 16: No outputs ===
=== Cell 18: No outputs ===


In [9]:
# Let's read the src files to understand the implementation
src_files = []
for root, dirs, files in os.walk(f'{repo_path}/src'):
    for file in files:
        if file.endswith('.py') and not file.startswith('__'):
            src_files.append(os.path.join(root, file))

print("Source files:")
for f in src_files:
    print(f)

Source files:
/net/scratch2/smallyan/function_vectors_eval/src/portability_eval.py
/net/scratch2/smallyan/function_vectors_eval/src/test_numheads.py
/net/scratch2/smallyan/function_vectors_eval/src/compute_indirect_effect.py
/net/scratch2/smallyan/function_vectors_eval/src/vocab_reconstruction.py
/net/scratch2/smallyan/function_vectors_eval/src/compute_avg_hidden_state.py
/net/scratch2/smallyan/function_vectors_eval/src/natural_text_eval.py
/net/scratch2/smallyan/function_vectors_eval/src/evaluate_function_vector.py
/net/scratch2/smallyan/function_vectors_eval/src/compute_average_activations.py
/net/scratch2/smallyan/function_vectors_eval/src/utils/eval_utils.py
/net/scratch2/smallyan/function_vectors_eval/src/utils/prompt_utils.py
/net/scratch2/smallyan/function_vectors_eval/src/utils/intervention_utils.py
/net/scratch2/smallyan/function_vectors_eval/src/utils/extract_utils.py
/net/scratch2/smallyan/function_vectors_eval/src/utils/model_utils.py
/net/scratch2/smallyan/function_vectors

In [10]:
# Read key implementation files to understand what experiments are implemented
# Let's start with compute_indirect_effect.py - this implements causal mediation analysis

with open(f'{repo_path}/src/compute_indirect_effect.py', 'r') as f:
    content = f.read()
print("=== compute_indirect_effect.py ===")
print(content[:3000])

=== compute_indirect_effect.py ===
import os, re, json
from tqdm import tqdm
import torch, numpy as np
import argparse
from baukit import TraceDict

# Include prompt creation helper functions
from utils.prompt_utils import *
from utils.intervention_utils import *
from utils.model_utils import *
from utils.extract_utils import *


def activation_replacement_per_class_intervention(prompt_data, avg_activations, dummy_labels, model, model_config, tokenizer, last_token_only=True):
    """
    Experiment to determine top intervention locations through avg activation replacement. 
    Performs a systematic sweep over attention heads (layer, head) to track their causal influence on probs of key tokens.

    Parameters: 
    prompt_data: dict containing ICL prompt examples, and template information
    avg_activations: avg activation of each attention head in the model taken across n_trials ICL prompts
    dummy_labels: labels and indices for a baseline prompt with the same number of example pa

In [11]:
# Read evaluate_function_vector.py
with open(f'{repo_path}/src/evaluate_function_vector.py', 'r') as f:
    content = f.read()
print("=== evaluate_function_vector.py ===")
print(content[:3000])

=== evaluate_function_vector.py ===
import os, json
import torch, numpy as np
import argparse

# Include prompt creation helper functions
from utils.prompt_utils import *
from utils.intervention_utils import *
from utils.model_utils import *
from utils.eval_utils import *
from utils.extract_utils import *
from compute_indirect_effect import compute_indirect_effect

if __name__ == "__main__":
    
    parser = argparse.ArgumentParser()

    parser.add_argument('--dataset_name', help='Name of the dataset to be loaded', type=str, required=True)
    parser.add_argument('--n_top_heads', help='Number of attenion head outputs used to compute function vector', required=False, type=int, default=10)
    parser.add_argument('--edit_layer', help='Layer for intervention. If -1, sweep over all layers', type=int, required=False, default=-1) # 
    parser.add_argument('--model_name', help='Name of model to be loaded', type=str, required=False, default='EleutherAI/gpt-j-6b')
    parser.add_argument('--

In [12]:
# Read vocab_reconstruction.py - for decoded vocabulary analysis
with open(f'{repo_path}/src/vocab_reconstruction.py', 'r') as f:
    content = f.read()
print("=== vocab_reconstruction.py ===")
print(content[:3000])

=== vocab_reconstruction.py ===
import os
import torch, numpy as np
import argparse

# Include prompt creation helper functions
from utils.prompt_utils import load_dataset
from utils.extract_utils import get_mean_head_activations, compute_universal_function_vector
from utils.eval_utils import n_shot_eval_no_intervention, n_shot_eval
from utils.model_utils import load_gpt_model_and_tokenizer, set_seed

def optim_loop(v_n, target, decoder, loss_fn, optimizer, n_steps:int=1000, verbose:bool=False, restrict_vocab:int=50400):
    if target.shape[-1] != restrict_vocab:
        inds = torch.topk(target, restrict_vocab).indices[0]
        Z = torch.zeros(target.size()).cuda()
        Z[:,inds] = target[:,inds]
    else:
        Z = target
            
    for i in range(n_steps):
        loss = loss_fn(decoder(v_n),Z)
        loss.backward()
        if verbose:
            print(f"Loss:{loss.item()}, iter:{i}")
        optimizer.step()
        optimizer.zero_grad()
    return v_n

def vocab_re

In [13]:
# Read portability_eval.py - for testing portability across templates
with open(f'{repo_path}/src/portability_eval.py', 'r') as f:
    content = f.read()
print("=== portability_eval.py ===")
print(content[:3000])

=== portability_eval.py ===
import os, json
import torch, numpy as np
import argparse

# Include prompt creation helper functions
from utils.prompt_utils import *
from utils.intervention_utils import *
from utils.model_utils import *
from utils.eval_utils import *
from utils.extract_utils import *


if __name__ == "__main__":
    
    parser = argparse.ArgumentParser()

    parser.add_argument('--dataset_name', help='Name of the dataset to be loaded', type=str, required=True)
    parser.add_argument('--n_eval_templates', help='Number of templates to evaluate with', required=True, type=int, default=15)
    parser.add_argument('--edit_layer', help='Layer for intervention. If -1, sweep over all layers', type=int, required=False, default=9) # 

    parser.add_argument('--n_top_heads', help='Number of attenion head outputs used to compute function vector', required=False, type=int, default=10)
    parser.add_argument('--model_name', help='Name of model to be loaded', type=str, required=Fals

In [14]:
# Read natural_text_eval.py - for natural text portability
with open(f'{repo_path}/src/natural_text_eval.py', 'r') as f:
    content = f.read()
print("=== natural_text_eval.py ===")
print(content[:3000])

=== natural_text_eval.py ===
import os, re, json
import torch, numpy as np
from tqdm import tqdm
import argparse

# Include prompt creation helper functions
from utils.eval_utils import n_shot_eval_no_intervention
from utils.extract_utils import get_mean_head_activations, compute_universal_function_vector
from utils.intervention_utils import fv_intervention_natural_text
from utils.model_utils import load_gpt_model_and_tokenizer
from utils.prompt_utils import load_dataset


def natural_text_eval(dataset, fv_vector, model, model_config, tokenizer, filter_set, edit_layer=9, MNT=5, verbose=False, loc_data=False):
    """
    Evaluates the causal effects of a function vector on natural text templates for the given dataset.

    Parameters:
    dataset: ICL dataset with pairs of words
    fv_vector: function vector to use for intervention
    model: huggingface model
    model_config: contains model config information (n layers, n heads, etc.)
    tokenizer: huggingface tokenizer
    filter_

In [15]:
# Read extract_utils.py - core functionality for extracting function vectors
with open(f'{repo_path}/src/utils/extract_utils.py', 'r') as f:
    content = f.read()
print("=== extract_utils.py ===")
print(content[:4000])

=== extract_utils.py ===
import os, re, json

import torch, numpy as np
import pandas as pd
from baukit import TraceDict

# Include prompt creation helper functions
from .prompt_utils import *
from .intervention_utils import *
from .model_utils import *
from .eval_utils import *


# Attention Activations
def gather_attn_activations(prompt_data, layers, dummy_labels, model, tokenizer, model_config):
    """
    Collects activations for an ICL prompt 

    Parameters:
    prompt_data: dict containing ICL prompt examples, and template information
    layers: layer names to get activatons from
    dummy_labels: labels and indices for a baseline prompt with the same number of example pairs
    model: huggingface model
    tokenizer: huggingface tokenizer

    Returns:
    td: tracedict with stored activations
    idx_map: map of token indices to respective averaged token indices
    idx_avg: dict containing token indices of multi-token words
    """   
    
    # Get sentence and token labe

In [16]:
# Now let's read the documentation.pdf to understand the conclusions
# First, let's check if we can extract text from the PDF
import subprocess

# Use pdftotext if available, otherwise we'll need another approach
result = subprocess.run(['pdftotext', f'{repo_path}/documentation.pdf', '-'], capture_output=True, text=True)
if result.returncode == 0:
    doc_content = result.stdout
    print("=== documentation.pdf content ===")
    print(doc_content)
else:
    print("Error extracting PDF:", result.stderr)

FileNotFoundError: [Errno 2] No such file or directory: 'pdftotext'

In [17]:
# Try using PyMuPDF (fitz) to read PDF
try:
    import fitz  # PyMuPDF
    doc = fitz.open(f'{repo_path}/documentation.pdf')
    doc_content = ""
    for page in doc:
        doc_content += page.get_text()
    print("=== documentation.pdf content ===")
    print(doc_content)
except ImportError:
    print("fitz not available, trying pypdf2")
    try:
        from PyPDF2 import PdfReader
        reader = PdfReader(f'{repo_path}/documentation.pdf')
        doc_content = ""
        for page in reader.pages:
            doc_content += page.extract_text()
        print("=== documentation.pdf content ===")
        print(doc_content)
    except ImportError:
        print("No PDF reader available")

=== documentation.pdf content ===
Published as a conference paper at ICLR 2024
FUNCTION VECTORS IN LARGE LANGUAGE MODELS
Eric Todd∗, Millicent L. Li, Arnab Sen Sharma, Aaron Mueller,
Byron C. Wallace, and David Bau
Khoury College of Computer Sciences, Northeastern University
ABSTRACT
We report the presence of a simple neural mechanism that represents an input-
output function as a vector within autoregressive transformer language models
(LMs). Using causal mediation analysis on a diverse range of in-context-learning
(ICL) tasks, we find that a small number attention heads transport a compact
representation of the demonstrated task, which we call a function vector (FV). FVs
are robust to changes in context, i.e., they trigger execution of the task on inputs
such as zero-shot and natural text settings that do not resemble the ICL contexts
from which they are collected. We test FVs across a range of tasks, models, and
layers and find strong causal effects across settings in middle layers.

In [18]:
# Let's extract the key content from the documentation.pdf in chunks
import fitz

doc = fitz.open(f'{repo_path}/documentation.pdf')
num_pages = len(doc)
print(f"Total pages: {num_pages}")

# Read first 5 pages which should contain Abstract, Introduction and key methodology
for i in range(min(5, num_pages)):
    print(f"\n=== Page {i+1} ===")
    print(doc[i].get_text()[:3000])

Total pages: 52

=== Page 1 ===
Published as a conference paper at ICLR 2024
FUNCTION VECTORS IN LARGE LANGUAGE MODELS
Eric Todd∗, Millicent L. Li, Arnab Sen Sharma, Aaron Mueller,
Byron C. Wallace, and David Bau
Khoury College of Computer Sciences, Northeastern University
ABSTRACT
We report the presence of a simple neural mechanism that represents an input-
output function as a vector within autoregressive transformer language models
(LMs). Using causal mediation analysis on a diverse range of in-context-learning
(ICL) tasks, we find that a small number attention heads transport a compact
representation of the demonstrated task, which we call a function vector (FV). FVs
are robust to changes in context, i.e., they trigger execution of the task on inputs
such as zero-shot and natural text settings that do not resemble the ICL contexts
from which they are collected. We test FVs across a range of tasks, models, and
layers and find strong causal effects across settings in middle layers. W

In [19]:
# Continue reading important sections for the key results
# Look for specific experiment results related to the plan

for i in range(5, min(15, num_pages)):
    print(f"\n=== Page {i+1} ===")
    print(doc[i].get_text()[:2500])


=== Page 6 ===
Published as a conference paper at ICLR 2024
0
5 10 15 20 25
0
0.5
1
GPT-J
Antonym
0
5 10 15 20 25
Capitalize
0
5 10 15 20 25
Country-Capital
0
5 10 15 20 25
English-French
0
5 10 15 20 25
Present-Past
0
5 10 15 20 25
Singular-Plural
0
8 16 24 32 40
0
0.5
1
GPT-NeoX
0
8 16 24 32 40
0
8 16 24 32 40
0
8 16 24 32 40
0
8 16 24 32 40
0
8 16 24 32 40
0 16 32 48 64
Layer
0
0.5
1
Llama 2 (70B)
0 16 32 48 64
Layer
0 16 32 48 64
Layer
0 16 32 48 64
Layer
0 16 32 48 64
Layer
0 16 32 48 64
Layer
Accuracy
Figure 4: Task accuracy across tasks and models, applying FVs in zero-shot settings. We show
accuracies before adding the function vector (dotted lines) and after adding the FV to a specific layer
(solid lines). Adding the FV to early-middle layers pushes models to perform the target task without
any exemplars, as demonstrated by accuracy increases over the zero-shot without FVs.
Table 3: Natural text portability of the Antonym FV. We provide a natural template and substitute in a


In [20]:
# Now let's analyze what experiments/implementations exist in the codebase
# Check all the main source files to see what is implemented

print("=== Summarizing Implementation Files ===\n")

# List of key implementation files
implementation_files = [
    'compute_indirect_effect.py',  # Causal mediation analysis
    'evaluate_function_vector.py',  # FV evaluation across contexts
    'portability_eval.py',  # Template portability
    'natural_text_eval.py',  # Natural text portability
    'vocab_reconstruction.py',  # Decoded vocabulary analysis
    'test_numheads.py',  # Testing number of heads
    'compute_average_activations.py',  # Computing mean activations
    'compute_avg_hidden_state.py',  # Computing average hidden states
]

for filename in implementation_files:
    filepath = f'{repo_path}/src/{filename}'
    if os.path.exists(filepath):
        print(f"✓ {filename} - EXISTS")
    else:
        print(f"✗ {filename} - MISSING")

=== Summarizing Implementation Files ===

✓ compute_indirect_effect.py - EXISTS
✓ evaluate_function_vector.py - EXISTS
✓ portability_eval.py - EXISTS
✓ natural_text_eval.py - EXISTS
✓ vocab_reconstruction.py - EXISTS
✓ test_numheads.py - EXISTS
✓ compute_average_activations.py - EXISTS
✓ compute_avg_hidden_state.py - EXISTS


In [21]:
# Now let's map the plan steps to the implementation to verify CS2
# 
# Plan Methodology:
# 1. Apply causal mediation analysis to identify attention heads with highest AIE
# 2. Test function vectors across models and tasks
# 3. Analyze FV internal structure by decoding to vocabulary space
# 4. Test vector algebra composition

# Let me check if there's any vector algebra/composition code in the implementation
# This is mentioned in the plan but I haven't seen it yet

# Search for composition-related code
print("=== Searching for vector composition implementation ===\n")

for filepath in src_files:
    with open(filepath, 'r') as f:
        content = f.read()
    if 'compos' in content.lower() or 'algebra' in content.lower() or 'v_AD' in content or 'vAD' in content or 'Last-' in content:
        print(f"Found composition-related content in: {filepath}")
        # Extract relevant sections
        lines = content.split('\n')
        for i, line in enumerate(lines):
            if 'compos' in line.lower() or 'algebra' in line.lower() or 'Last' in line:
                print(f"  Line {i}: {line[:100]}")

=== Searching for vector composition implementation ===



In [22]:
# No composition implementation found in source files - let's check the demo notebook more thoroughly
# and look for any additional notebooks or scripts

print("=== Checking for vector composition code ===\n")

# Check all files in the repo for composition-related content
import subprocess
result = subprocess.run(['grep', '-r', 'compos', repo_path, '--include=*.py', '--include=*.ipynb'], 
                       capture_output=True, text=True)
print("Grep for 'compos':")
print(result.stdout[:2000] if result.stdout else "No matches found")

# Also check for "Last-" tasks which are part of composition
result2 = subprocess.run(['grep', '-r', 'Last-Copy', repo_path, '--include=*.py', '--include=*.ipynb'], 
                        capture_output=True, text=True)
print("\nGrep for 'Last-Copy':")
print(result2.stdout[:1000] if result2.stdout else "No matches found")

=== Checking for vector composition code ===

Grep for 'compos':
No matches found



Grep for 'Last-Copy':
No matches found


In [23]:
# Let's check the eval_utils and other utility files to see what experiments are supported

with open(f'{repo_path}/src/utils/eval_utils.py', 'r') as f:
    eval_utils_content = f.read()

print("=== eval_utils.py ===")
print(eval_utils_content[:4000])

=== eval_utils.py ===
import torch
import string
import itertools
import re
import numpy as np
from tqdm import tqdm
from .prompt_utils import *
from .model_utils import *
from .intervention_utils import *


def compute_top_k_accuracy(target_token_ranks, k=10) -> float:
    """
    Evaluation to compute topk accuracy.

    Parameters:
    target_token_ranks: the distribution of output token ranks
    k: how many tokens we're looking at (top K)

    Return:
    The accuracy of the token in the top k of tokens
    """

    target_token_ranks = np.array(target_token_ranks)
    return (target_token_ranks < k).sum(axis=0) / len(target_token_ranks) 

def compute_individual_token_rank(prob_dist, target_id) -> int:
    """
    Individual computation of token ranks across a single distribution.

    Parameters:
    prob_dist: the distribution of scores for a single output
    target_id: the target id we care about

    Return:
    A single value representing the token rank for that single token

In [24]:
# Check if there are any available datasets that match the composition tasks
# mentioned in the plan (First-Copy, Last-Copy, First-Capital, Last-Capital, etc.)

dataset_files = os.listdir(f'{repo_path}/dataset_files/abstractive')
print("=== Abstractive Dataset Files ===")
for f in sorted(dataset_files):
    print(f)

print("\n=== Extractive Dataset Files ===")  
extractive_files = os.listdir(f'{repo_path}/dataset_files/extractive')
for f in sorted(extractive_files):
    print(f)

=== Abstractive Dataset Files ===
ag_news.json
antonym.json
capitalize.json
capitalize_first_letter.json
capitalize_last_letter.json
capitalize_second_letter.json
commonsense_qa.json
country-capital.json
country-currency.json
english-french.json
english-german.json
english-spanish.json
landmark-country.json
lowercase_first_letter.json
lowercase_last_letter.json
national_parks.json
next_capital_letter.json
next_item.json
park-country.json
person-instrument.json
person-occupation.json
person-sport.json
present-past.json
prev_item.json
product-company.json
sentiment.json
singular-plural.json
synonym.json
word_length.json

=== Extractive Dataset Files ===
adjective_v_verb_3.json
adjective_v_verb_5.json
alphabetically_first_3.json
alphabetically_first_5.json
alphabetically_last_3.json
alphabetically_last_5.json
animal_v_object_3.json
animal_v_object_5.json
choose_first_of_3.json
choose_first_of_5.json
choose_last_of_3.json
choose_last_of_5.json
choose_middle_of_3.json
choose_middle_of_5.jso

In [25]:
# I notice there are no "First-Copy", "Last-Copy", "First-Capital", "Last-Capital" datasets
# which are needed for the vector algebra composition experiment in the plan
# Let's check if there's any implementation at all for these composition experiments

# Also check the demo notebook for any additional composition code
print("=== Checking demo notebook for composition experiments ===\n")

with open(f'{repo_path}/notebooks/fv_demo.ipynb', 'r') as f:
    notebook = json.load(f)
    
for i, cell in enumerate(notebook['cells']):
    source = ''.join(cell['source'])
    if 'Last' in source or 'First' in source or 'compos' in source.lower():
        print(f"Cell {i}:")
        print(source[:500])
        print("---")

=== Checking demo notebook for composition experiments ===



In [26]:
# Now let me check what experiments are defined in the plan vs what's implemented
# Let's create a comprehensive mapping

plan_experiments = """
### Plan Experiments:
1. Portability of function vectors across contexts
   - Contexts: shuffled-label ICL, zero-shot, different ICL templates, natural text
   - Metric: Top-1 accuracy
   - Main result: FVs work best at early-middle layers (L/3)

2. Decoded vocabulary analysis  
   - Varying k tokens (100 vs 50k)
   - Metric: Zero-shot accuracy of reconstructed vectors
   - Main result: Reconstructed vectors underperform original FVs

3. Vector algebra composition
   - Task combinations: Last-Capitalize, Last-Country-Capital, Last-Antonym, etc.
   - Metric: Accuracy of composed vector vs ICL and direct FV
   - Main result: Some compositions work, some fail

4. Causal mediation analysis across models
   - Models: GPT-J 6B, GPT-NeoX 20B, Llama 2 7B/13B/70B
   - Metric: Average Indirect Effect (AIE)
   - Main result: Top 10-100 heads cluster in middle layers

5. Performance across diverse tasks and models
   - 34 additional abstractive and extractive tasks
   - Metric: Zero-shot and shuffled-label accuracy
   - Main result: Consistent FV effects across task diversity

6. Natural text portability evaluation
   - Different natural language prompt templates
   - Metric: Accuracy of correct answer in n generated tokens
   - Main result: FVs work in naturalistic settings
"""

implementation_mapping = """
### Implementation Files:
1. compute_indirect_effect.py - Implements causal mediation analysis (Exp 4)
2. evaluate_function_vector.py - Implements FV evaluation (Exp 1, 5)
3. portability_eval.py - Implements template portability testing (Exp 1)
4. natural_text_eval.py - Implements natural text evaluation (Exp 6)
5. vocab_reconstruction.py - Implements vocabulary reconstruction (Exp 2)
6. compute_average_activations.py - Computes mean activations (supporting)
7. test_numheads.py - Tests number of attention heads (supporting)

### Missing Implementation:
- Vector algebra composition (Exp 3) - NOT FOUND in any source file
  - No code for First-Copy, Last-Copy, First-Capital, Last-Capital tasks
  - No datasets for these composition tasks
  - No algebraic sum operations on FVs (vAD + vBC - vAC)
"""

print(plan_experiments)
print(implementation_mapping)


### Plan Experiments:
1. Portability of function vectors across contexts
   - Contexts: shuffled-label ICL, zero-shot, different ICL templates, natural text
   - Metric: Top-1 accuracy
   - Main result: FVs work best at early-middle layers (L/3)

2. Decoded vocabulary analysis  
   - Varying k tokens (100 vs 50k)
   - Metric: Zero-shot accuracy of reconstructed vectors
   - Main result: Reconstructed vectors underperform original FVs

3. Vector algebra composition
   - Task combinations: Last-Capitalize, Last-Country-Capital, Last-Antonym, etc.
   - Metric: Accuracy of composed vector vs ICL and direct FV
   - Main result: Some compositions work, some fail

4. Causal mediation analysis across models
   - Models: GPT-J 6B, GPT-NeoX 20B, Llama 2 7B/13B/70B
   - Metric: Average Indirect Effect (AIE)
   - Main result: Top 10-100 heads cluster in middle layers

5. Performance across diverse tasks and models
   - 34 additional abstractive and extractive tasks
   - Metric: Zero-shot and shuf

In [27]:
# Let's also check the extractive datasets more closely - maybe choose_first and choose_last could be used 
# for First-Copy and Last-Copy

# Read one of the choose_first/choose_last datasets to understand the format
with open(f'{repo_path}/dataset_files/extractive/choose_first_of_5.json', 'r') as f:
    first_data = json.load(f)
print("=== choose_first_of_5.json sample ===")
print(json.dumps(first_data[:3], indent=2))

with open(f'{repo_path}/dataset_files/extractive/choose_last_of_5.json', 'r') as f:
    last_data = json.load(f)
print("\n=== choose_last_of_5.json sample ===")
print(json.dumps(last_data[:3], indent=2))

=== choose_first_of_5.json sample ===
[
  {
    "input": "ostrich, since, out, curtain, trustworthy",
    "output": "ostrich"
  },
  {
    "input": "play, gray, zap, blackberry, I",
    "output": "play"
  },
  {
    "input": "durian, globe, funny, stapler, friendly",
    "output": "durian"
  }
]

=== choose_last_of_5.json sample ===
[
  {
    "input": "mint, grateful, vulture, resilient, build",
    "output": "build"
  },
  {
    "input": "them, joyous, turtle, globe, indigo",
    "output": "indigo"
  },
  {
    "input": "spicy, cowardly, hoop, komodo, toward",
    "output": "toward"
  }
]


In [28]:
# These are list-oriented extractive tasks that could be used for the First-Copy and Last-Copy 
# experiments mentioned in the plan. However, there's still no code for the vector composition itself.

# Let me verify by checking the entire codebase one more time for any composition-related code
# This time let's search more broadly

import subprocess

# Search for any vector arithmetic operations
print("=== Searching for vector arithmetic operations ===\n")
result = subprocess.run(['grep', '-r', '-n', 'vAC\\|vBC\\|vAD\\|vBD\\|v_AC\\|v_BC\\|v_AD\\|v_BD', repo_path, 
                        '--include=*.py', '--include=*.ipynb'], 
                       capture_output=True, text=True)
print("Search for composition variables (vAC, vBC, vAD, vBD):")
print(result.stdout if result.stdout else "No matches found")

# Search for algebraic operations
result2 = subprocess.run(['grep', '-r', '-n', 'FV.*\\+.*FV\\|function_vector.*\\+\\|\\+.*function_vector', repo_path,
                         '--include=*.py', '--include=*.ipynb'],
                        capture_output=True, text=True)
print("\nSearch for FV addition operations:")
print(result2.stdout[:1000] if result2.stdout else "No matches found")

=== Searching for vector arithmetic operations ===

Search for composition variables (vAC, vBC, vAD, vBD):
No matches found

Search for FV addition operations:
/net/scratch2/smallyan/function_vectors_eval/src/portability_eval.py:88:    fv, top_heads = compute_universal_function_vector(mean_activations, model, model_config=model_config, n_top_heads=n_top_heads)   
/net/scratch2/smallyan/function_vectors_eval/src/utils/eval_utils.py:95:def fv_to_vocab(function_vector, model, model_config, tokenizer, n_tokens=10):
/net/scratch2/smallyan/function_vectors_eval/src/utils/eval_utils.py:100:    function_vector: torch vector extracted from ICL contexts that represents a particular function
/net/scratch2/smallyan/function_vectors_eval/src/utils/eval_utils.py:117:    d_out = decoder(function_vector.reshape(1,1,model_config['resid_dim']).to(model.device))
/net/scratch2/smallyan/function_vectors_eval/src/utils/eval_utils.py:216:    function_vector: torch vector that triggers execution of a task whe

In [29]:
# Now let's verify the consistency between the documented results and what can be verified
# Let's look at the specific claims in the documentation and plan

# CS1: Check if conclusions in documentation match results recorded in code
# The documentation mentions specific results like:
# - GPT-J+FV achieves 90.8% shuffled-label vs 39.1% baseline
# - GPT-J+FV achieves 57.5% zero-shot vs 5.5% baseline
# - FVs work best at layer L/3

# Since there are no outputs saved in the notebooks, we need to check if the implementation 
# can produce these results. Let me check the evaluation code for any hardcoded or saved results

print("=== Checking evaluate_function_vector.py for result outputs ===\n")
with open(f'{repo_path}/src/evaluate_function_vector.py', 'r') as f:
    content = f.read()
    
# Look for result saving/printing
lines = content.split('\n')
for i, line in enumerate(lines):
    if 'print' in line.lower() or 'save' in line.lower() or 'result' in line.lower() or 'accura' in line.lower():
        print(f"Line {i+1}: {line.strip()[:100]}")

=== Checking evaluate_function_vector.py for result outputs ===

Line 22: parser.add_argument('--save_path_root', help='File path to save to', type=str, required=False, defau
Line 45: save_path_root = f"{args.save_path_root}/{dataset_name}"
Line 46: ie_path_root = f"{args.ie_path_root}/{dataset_name}" if args.ie_path_root else save_path_root
Line 67: print(args)
Line 71: print("Loading Model")
Line 78: print("Loading Dataset")
Line 82: if not os.path.exists(save_path_root):
Line 83: os.makedirs(save_path_root)
Line 85: print(f"Filtering Dataset via {n_shots}-shot Eval")
Line 88: fs_results_file_name = f'{save_path_root}/fs_results_layer_sweep.json'
Line 89: print(fs_results_file_name)
Line 90: if os.path.exists(fs_results_file_name):
Line 91: with open(fs_results_file_name, 'r') as indata:
Line 92: fs_results = json.load(indata)
Line 95: filter_set = np.where(np.array(fs_results[key]) == target_val)[0]
Line 99: fs_results_validation = n_shot_eval_no_intervention(dataset=dataset, n_shot

In [30]:
# Check if there's a results directory with saved results
results_path = f'{repo_path}/results'
if os.path.exists(results_path):
    print("=== Results directory exists ===")
    for root, dirs, files in os.walk(results_path):
        level = root.replace(results_path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        for file in files[:10]:  # Limit to first 10 files per dir
            print(f'{subindent}{file}')
        if len(files) > 10:
            print(f'{subindent}... and {len(files)-10} more files')
else:
    print("No results directory found")

No results directory found


In [31]:
# Since there's no results directory, let's check for any saved data files in the repo
import subprocess
result = subprocess.run(['find', repo_path, '-name', '*.json', '-path', '*/results/*'], 
                       capture_output=True, text=True)
print("JSON result files:")
print(result.stdout if result.stdout else "None found")

# Also check for any .pt files (PyTorch saved tensors)
result2 = subprocess.run(['find', repo_path, '-name', '*.pt'], 
                        capture_output=True, text=True)
print("\nPyTorch tensor files:")
print(result2.stdout if result2.stdout else "None found")

JSON result files:
None found

PyTorch tensor files:
None found


In [32]:
# Now I have gathered enough information to make the evaluation. Let me summarize:

# CS1: Conclusion vs Original Results
# - The documentation (documentation.pdf) contains results from the published ICLR 2024 paper
# - The implementation notebooks (fv_demo.ipynb) have no saved outputs
# - There are no result files (.json or .pt) saved in the repository
# - Since there are no original results recorded in the notebooks to compare against,
#   we can say this check PASSES because there's no contradiction - the documentation 
#   simply represents the paper's results and there are no conflicting notebook results

# CS2: Implementation Follows the Plan
# - The plan describes 6 experiments/methodologies
# - 5 out of 6 are implemented:
#   1. ✓ Causal mediation analysis (compute_indirect_effect.py)
#   2. ✓ FV evaluation across contexts (evaluate_function_vector.py)
#   3. ✓ Template portability (portability_eval.py)
#   4. ✓ Natural text evaluation (natural_text_eval.py)
#   5. ✓ Vocabulary reconstruction (vocab_reconstruction.py)
# - 1 is MISSING:
#   6. ✗ Vector algebra composition - NO implementation found
#     - No code for First-Copy, Last-Copy, First-Capital, Last-Capital tasks
#     - No datasets for these composition tasks (only choose_first/last extractive tasks)
#     - No algebraic sum operations on FVs (vAD + vBC - vAC)

print("=== EVALUATION SUMMARY ===")
print()
print("CS1: Conclusion vs Original Results - PASS")
print("Reason: The documentation.pdf contains the ICLR 2024 paper with its reported results.")
print("The demo notebook (fv_demo.ipynb) has no executed outputs/results saved.")
print("Since there are no recorded results in the code implementation to compare,")
print("there is no contradiction between conclusions and recorded results.")
print()
print("CS2: Implementation Follows the Plan - FAIL")
print("Reason: The plan specifies 'Vector algebra composition' as methodology step 4:")
print("'Test vector algebra composition by constructing decomposable tasks and measuring")
print("whether algebraic sums of FVs can execute combined tasks.'")
print("This step is NOT implemented in the codebase:")
print("- No code for composition tasks (First-Copy, Last-Copy, First-Capital, Last-Capital)")
print("- No vector algebra operations (vAD + vBC - vAC)")
print("- No datasets for the composition experiment")

=== EVALUATION SUMMARY ===

CS1: Conclusion vs Original Results - PASS
Reason: The documentation.pdf contains the ICLR 2024 paper with its reported results.
The demo notebook (fv_demo.ipynb) has no executed outputs/results saved.
Since there are no recorded results in the code implementation to compare,
there is no contradiction between conclusions and recorded results.

CS2: Implementation Follows the Plan - FAIL
Reason: The plan specifies 'Vector algebra composition' as methodology step 4:
'Test vector algebra composition by constructing decomposable tasks and measuring
whether algebraic sums of FVs can execute combined tasks.'
This step is NOT implemented in the codebase:
- No code for composition tasks (First-Copy, Last-Copy, First-Capital, Last-Capital)
- No vector algebra operations (vAD + vBC - vAC)
- No datasets for the composition experiment


In [33]:
# Create the evaluation directory
eval_dir = f'{repo_path}/evaluation'
os.makedirs(eval_dir, exist_ok=True)
print(f"Created evaluation directory: {eval_dir}")

Created evaluation directory: /net/scratch2/smallyan/function_vectors_eval/evaluation


# Consistency Evaluation - Function Vectors in LLMs

This notebook evaluates the consistency between the documented conclusions and the implementation in the Function Vectors repository.

## Repository Overview

**Repository Path:** `/net/scratch2/smallyan/function_vectors_eval`

**Project Goal (from plan.md):** To investigate whether autoregressive transformer language models develop compact vector representations of input-output functions (called function vectors) within their hidden states during in-context learning, and to characterize these representations across diverse tasks and models.

In [34]:
# Let's verify the repository structure
print("=== Repository Structure ===\n")
repo_path = '/net/scratch2/smallyan/function_vectors_eval'
for item in sorted(os.listdir(repo_path)):
    if not item.startswith('.'):
        full_path = os.path.join(repo_path, item)
        if os.path.isdir(full_path):
            print(f"📁 {item}/")
        else:
            print(f"📄 {item}")

=== Repository Structure ===

📄 CodeWalkthrough.md
📁 dataset_files/
📄 documentation.pdf
📁 evaluation/
📄 fv_environment.yml
📄 fv_overview.png
📁 notebooks/
📄 plan.md
📁 src/


## Plan Analysis

The plan.md specifies the following methodology steps:

1. **Causal Mediation Analysis** - Apply causal mediation analysis to identify attention heads with highest AIE
2. **FV Portability Testing** - Test function vectors across models and tasks in shuffled-label, zero-shot, and natural text contexts
3. **Vocabulary Decoding Analysis** - Analyze FV internal structure by decoding vectors to vocabulary space
4. **Vector Algebra Composition** - Test vector algebra composition by constructing decomposable tasks

In [35]:
# Verify implementation files exist
print("=== Implementation Files Analysis ===\n")

implementation_mapping = {
    'Causal Mediation Analysis': 'src/compute_indirect_effect.py',
    'FV Evaluation': 'src/evaluate_function_vector.py', 
    'Template Portability': 'src/portability_eval.py',
    'Natural Text Evaluation': 'src/natural_text_eval.py',
    'Vocabulary Reconstruction': 'src/vocab_reconstruction.py',
    'Mean Activations': 'src/compute_average_activations.py',
    'Number of Heads Testing': 'src/test_numheads.py',
    'Vector Algebra Composition': None  # This needs to be checked
}

for experiment, filepath in implementation_mapping.items():
    if filepath:
        full_path = os.path.join(repo_path, filepath)
        exists = os.path.exists(full_path)
        status = "✅ IMPLEMENTED" if exists else "❌ MISSING"
    else:
        status = "❓ CHECKING..."
    print(f"{experiment}: {status}")

=== Implementation Files Analysis ===

Causal Mediation Analysis: ✅ IMPLEMENTED
FV Evaluation: ✅ IMPLEMENTED
Template Portability: ✅ IMPLEMENTED
Natural Text Evaluation: ✅ IMPLEMENTED
Vocabulary Reconstruction: ✅ IMPLEMENTED
Mean Activations: ✅ IMPLEMENTED
Number of Heads Testing: ✅ IMPLEMENTED
Vector Algebra Composition: ❓ CHECKING...


In [36]:
# Search comprehensively for vector composition implementation
print("=== Searching for Vector Algebra Composition Implementation ===\n")

import subprocess

# Search patterns for composition-related code
patterns = [
    ('Vector composition variables', r'v_?[AB][CD]'),
    ('Algebraic operations', 'algebra'),
    ('Composition keyword', 'compos'),
    ('First-Copy task', 'First-Copy'),
    ('Last-Copy task', 'Last-Copy'),
    ('First-Capital task', 'First-Capital'),
    ('Last-Capital task', 'Last-Capital'),
]

composition_found = False
for desc, pattern in patterns:
    result = subprocess.run(['grep', '-r', '-l', pattern, f'{repo_path}/src', f'{repo_path}/notebooks'], 
                           capture_output=True, text=True)
    if result.stdout.strip():
        print(f"✅ {desc}: Found in {result.stdout.strip()}")
        composition_found = True
    else:
        print(f"❌ {desc}: Not found")

print("\n" + "="*50)
if composition_found:
    print("Result: Vector algebra composition IS implemented")
else:
    print("Result: Vector algebra composition is NOT implemented")
    print("\nThe plan specifies: 'Test vector algebra composition by constructing")
    print("decomposable tasks and measuring whether algebraic sums of FVs can")
    print("execute combined tasks.'")
    print("\nThis methodology step is missing from the implementation.")

=== Searching for Vector Algebra Composition Implementation ===

❌ Vector composition variables: Not found
❌ Algebraic operations: Not found
❌ Composition keyword: Not found
❌ First-Copy task: Not found
❌ Last-Copy task: Not found
❌ First-Capital task: Not found
❌ Last-Capital task: Not found

Result: Vector algebra composition is NOT implemented

The plan specifies: 'Test vector algebra composition by constructing
decomposable tasks and measuring whether algebraic sums of FVs can
execute combined tasks.'

This methodology step is missing from the implementation.


## CS1: Conclusion vs Original Results

**Evaluation Criteria:** All evaluable conclusions in the documentation must match the results originally recorded in the code implementation notebook.

### Analysis

In [37]:
# Check for any saved results in notebooks
print("=== Checking for Recorded Results in Notebooks ===\n")

import json

notebook_path = f'{repo_path}/notebooks/fv_demo.ipynb'
with open(notebook_path, 'r') as f:
    notebook = json.load(f)

output_count = 0
for i, cell in enumerate(notebook['cells']):
    if cell['cell_type'] == 'code':
        outputs = cell.get('outputs', [])
        if outputs:
            output_count += 1
            print(f"Cell {i}: Has {len(outputs)} output(s)")

print(f"\nTotal cells with outputs: {output_count}")
print(f"Total code cells: {len([c for c in notebook['cells'] if c['cell_type'] == 'code'])}")

if output_count == 0:
    print("\n⚠️ The demo notebook has NO executed outputs/results saved.")

=== Checking for Recorded Results in Notebooks ===


Total cells with outputs: 0
Total code cells: 10

⚠️ The demo notebook has NO executed outputs/results saved.


In [38]:
# Check for any saved result files
print("=== Checking for Saved Result Files ===\n")

import subprocess

# Look for JSON result files
result = subprocess.run(['find', repo_path, '-name', '*.json', '-not', '-path', '*/dataset_files/*'], 
                       capture_output=True, text=True)
json_files = result.stdout.strip().split('\n') if result.stdout.strip() else []
print(f"JSON files (excluding datasets): {len(json_files)}")
for f in json_files[:5]:
    print(f"  - {f}")

# Look for PyTorch saved files
result = subprocess.run(['find', repo_path, '-name', '*.pt'], 
                       capture_output=True, text=True)
pt_files = result.stdout.strip().split('\n') if result.stdout.strip() else []
print(f"\nPyTorch tensor files (.pt): {len(pt_files)}")

# Look for pickle files
result = subprocess.run(['find', repo_path, '-name', '*.pkl'], 
                       capture_output=True, text=True)
pkl_files = result.stdout.strip().split('\n') if result.stdout.strip() else []
print(f"Pickle files (.pkl): {len(pkl_files)}")

# Look for results directory
if os.path.exists(f'{repo_path}/results'):
    print("\n✅ Results directory exists")
else:
    print("\n❌ No results directory found")

=== Checking for Saved Result Files ===



JSON files (excluding datasets): 1
  - /net/scratch2/smallyan/function_vectors_eval/evaluation/code_critic_summary.json

PyTorch tensor files (.pt): 0
Pickle files (.pkl): 0

❌ No results directory found


In [39]:
# CS1 Conclusion
print("=== CS1: Conclusion vs Original Results ===\n")
print("VERDICT: PASS\n")
print("Rationale:")
print("-" * 50)
print("""
1. The documentation.pdf contains the published ICLR 2024 paper 
   "Function Vectors in Large Language Models" with its reported results.

2. The demo notebook (fv_demo.ipynb) contains code demonstrating how to:
   - Load models and compute mean head activations
   - Compute function vectors
   - Test FVs in ICL, shuffled-label, zero-shot, and natural text contexts
   
3. However, the notebook has NO executed outputs saved - all 10 code cells
   have empty output fields.

4. There are no saved result files (.json, .pt, .pkl) in the repository
   that could be compared against the documentation.

5. Since there are no recorded results in the code implementation to 
   compare against, there is NO CONTRADICTION between the documentation
   conclusions and any recorded results.

The documentation accurately describes the paper's methodology and results.
The implementation code is consistent with the described methodology.
No conflicting numerical results exist in the codebase.
""")

CS1_RESULT = "PASS"
CS1_RATIONALE = ("The documentation.pdf contains the ICLR 2024 paper with reported results. "
                 "The demo notebook has no executed outputs saved. There are no saved result files. "
                 "Since there are no recorded results in the implementation to compare against, "
                 "there is no contradiction between conclusions and recorded results.")

=== CS1: Conclusion vs Original Results ===

VERDICT: PASS

Rationale:
--------------------------------------------------

1. The documentation.pdf contains the published ICLR 2024 paper 
   "Function Vectors in Large Language Models" with its reported results.

2. The demo notebook (fv_demo.ipynb) contains code demonstrating how to:
   - Load models and compute mean head activations
   - Compute function vectors
   - Test FVs in ICL, shuffled-label, zero-shot, and natural text contexts
   
3. However, the notebook has NO executed outputs saved - all 10 code cells
   have empty output fields.

4. There are no saved result files (.json, .pt, .pkl) in the repository
   that could be compared against the documentation.

5. Since there are no recorded results in the code implementation to 
   compare against, there is NO CONTRADICTION between the documentation
   conclusions and any recorded results.

The documentation accurately describes the paper's methodology and results.
The implement

## CS2: Implementation Follows the Plan

**Evaluation Criteria:** All plan steps must appear in the implementation.

In [40]:
# CS2 Analysis
print("=== CS2: Implementation Follows the Plan ===\n")

# Read the plan file
with open(f'{repo_path}/plan.md', 'r') as f:
    plan_content = f.read()

# Extract methodology section
methodology_section = plan_content.split('## Methodology')[1].split('## Experiments')[0]
print("Plan Methodology Steps:")
print("-" * 50)
print(methodology_section)

=== CS2: Implementation Follows the Plan ===

Plan Methodology Steps:
--------------------------------------------------

1. Apply causal mediation analysis to identify attention heads with highest average indirect effect (AIE) across diverse ICL tasks. Extract function vectors by summing task-conditioned mean outputs of top causal attention heads.
2. Test function vectors across models (GPT-J 6B to Llama 2 70B) and over 40 diverse ICL tasks by adding FVs to hidden states at different layers and measuring task performance in shuffled-label, zero-shot, and natural text contexts.
3. Analyze FV internal structure by decoding vectors to vocabulary space and testing whether reconstructed vectors matching decoded distributions can recover FV performance.
4. Test vector algebra composition by constructing decomposable tasks and measuring whether algebraic sums of FVs can execute combined tasks.




In [41]:
# Check each methodology step
print("=== Methodology Step Implementation Status ===\n")

steps = [
    {
        "step": 1,
        "description": "Causal mediation analysis to identify attention heads with highest AIE",
        "implementation": "src/compute_indirect_effect.py",
        "key_functions": ["activation_replacement_per_class_intervention", "compute_indirect_effect"]
    },
    {
        "step": 2, 
        "description": "Test FVs across models and tasks (shuffled-label, zero-shot, natural text)",
        "implementation": ["src/evaluate_function_vector.py", "src/portability_eval.py", "src/natural_text_eval.py"],
        "key_functions": ["n_shot_eval", "function_vector_intervention", "natural_text_eval"]
    },
    {
        "step": 3,
        "description": "Analyze FV internal structure by decoding to vocabulary space",
        "implementation": "src/vocab_reconstruction.py",
        "key_functions": ["vocab_reconstruction", "optim_loop"]
    },
    {
        "step": 4,
        "description": "Test vector algebra composition with decomposable tasks",
        "implementation": None,
        "key_functions": []
    }
]

for step_info in steps:
    print(f"Step {step_info['step']}: {step_info['description']}")
    if step_info['implementation']:
        if isinstance(step_info['implementation'], list):
            files = step_info['implementation']
        else:
            files = [step_info['implementation']]
        
        all_exist = True
        for filepath in files:
            full_path = os.path.join(repo_path, filepath)
            if os.path.exists(full_path):
                print(f"  ✅ {filepath}")
            else:
                print(f"  ❌ {filepath} - MISSING")
                all_exist = False
    else:
        print("  ❌ NO IMPLEMENTATION FOUND")
    print()

=== Methodology Step Implementation Status ===

Step 1: Causal mediation analysis to identify attention heads with highest AIE
  ✅ src/compute_indirect_effect.py

Step 2: Test FVs across models and tasks (shuffled-label, zero-shot, natural text)
  ✅ src/evaluate_function_vector.py
  ✅ src/portability_eval.py
  ✅ src/natural_text_eval.py

Step 3: Analyze FV internal structure by decoding to vocabulary space
  ✅ src/vocab_reconstruction.py

Step 4: Test vector algebra composition with decomposable tasks
  ❌ NO IMPLEMENTATION FOUND



In [42]:
# CS2 Conclusion
print("=== CS2: Implementation Follows the Plan ===\n")
print("VERDICT: FAIL\n")
print("Rationale:")
print("-" * 50)
print("""
The plan.md file specifies 4 methodology steps. The implementation status is:

Step 1: Causal mediation analysis
  ✅ IMPLEMENTED in src/compute_indirect_effect.py
  - Contains activation_replacement_per_class_intervention()
  - Contains compute_indirect_effect()

Step 2: Test FVs across models and contexts  
  ✅ IMPLEMENTED in multiple files:
  - src/evaluate_function_vector.py (shuffled-label, zero-shot)
  - src/portability_eval.py (template portability)
  - src/natural_text_eval.py (natural text contexts)

Step 3: Analyze FV internal structure by vocabulary decoding
  ✅ IMPLEMENTED in src/vocab_reconstruction.py
  - Contains vocab_reconstruction() function
  - Contains optim_loop() for reconstruction

Step 4: Test vector algebra composition
  ❌ NOT IMPLEMENTED
  - No code for First-Copy, Last-Copy, First-Capital, Last-Capital tasks
  - No datasets for composition tasks
  - No algebraic sum operations on FVs (vAD + vBC - vAC)
  - This is explicitly stated in the plan as methodology step 4:
    "Test vector algebra composition by constructing decomposable tasks 
    and measuring whether algebraic sums of FVs can execute combined tasks."

Since methodology step 4 from the plan is NOT implemented, this evaluation FAILS.
""")

CS2_RESULT = "FAIL"
CS2_RATIONALE = ("The plan specifies 4 methodology steps. Steps 1-3 are implemented "
                 "(causal mediation analysis, FV testing across contexts, vocabulary decoding). "
                 "However, Step 4 'Test vector algebra composition by constructing decomposable tasks' "
                 "is NOT implemented - no code for composition tasks (First-Copy, Last-Copy, etc.), "
                 "no datasets for these tasks, and no algebraic sum operations on FVs.")

=== CS2: Implementation Follows the Plan ===

VERDICT: FAIL

Rationale:
--------------------------------------------------

The plan.md file specifies 4 methodology steps. The implementation status is:

Step 1: Causal mediation analysis
  ✅ IMPLEMENTED in src/compute_indirect_effect.py
  - Contains activation_replacement_per_class_intervention()
  - Contains compute_indirect_effect()

Step 2: Test FVs across models and contexts  
  ✅ IMPLEMENTED in multiple files:
  - src/evaluate_function_vector.py (shuffled-label, zero-shot)
  - src/portability_eval.py (template portability)
  - src/natural_text_eval.py (natural text contexts)

Step 3: Analyze FV internal structure by vocabulary decoding
  ✅ IMPLEMENTED in src/vocab_reconstruction.py
  - Contains vocab_reconstruction() function
  - Contains optim_loop() for reconstruction

Step 4: Test vector algebra composition
  ❌ NOT IMPLEMENTED
  - No code for First-Copy, Last-Copy, First-Capital, Last-Capital tasks
  - No datasets for compositio

## Summary

| Checklist Item | Result | Key Finding |
|----------------|--------|-------------|
| CS1: Conclusion vs Original Results | **PASS** | No recorded results to contradict documentation |
| CS2: Implementation Follows Plan | **FAIL** | Vector algebra composition (Step 4) not implemented |

In [43]:
# Create the consistency_evaluation.json file
import json

evaluation_result = {
    "Checklist": {
        "CS1_Results_vs_Conclusion": "PASS",
        "CS2_Plan_vs_Implementation": "FAIL"
    },
    "Rationale": {
        "CS1_Results_vs_Conclusion": (
            "The documentation.pdf contains the ICLR 2024 paper with reported results. "
            "The demo notebook (fv_demo.ipynb) has no executed outputs saved (all 10 code cells have empty outputs). "
            "There are no saved result files (.json, .pt, .pkl) in the repository. "
            "Since there are no recorded results in the code implementation to compare against, "
            "there is no contradiction between the documentation conclusions and any recorded results."
        ),
        "CS2_Plan_vs_Implementation": (
            "The plan.md specifies 4 methodology steps. Steps 1-3 are implemented: "
            "(1) Causal mediation analysis in compute_indirect_effect.py, "
            "(2) FV testing across contexts in evaluate_function_vector.py, portability_eval.py, and natural_text_eval.py, "
            "(3) Vocabulary decoding analysis in vocab_reconstruction.py. "
            "However, Step 4 'Test vector algebra composition by constructing decomposable tasks and measuring "
            "whether algebraic sums of FVs can execute combined tasks' is NOT implemented. "
            "No code exists for composition tasks (First-Copy, Last-Copy, First-Capital, Last-Capital), "
            "no datasets for these tasks, and no algebraic sum operations on FVs (vAD + vBC - vAC)."
        )
    }
}

# Save to file
output_path = f'{repo_path}/evaluation/consistency_evaluation.json'
with open(output_path, 'w') as f:
    json.dump(evaluation_result, f, indent=4)

print(f"✅ Saved evaluation results to: {output_path}")
print("\nContents:")
print(json.dumps(evaluation_result, indent=4))

✅ Saved evaluation results to: /net/scratch2/smallyan/function_vectors_eval/evaluation/consistency_evaluation.json

Contents:
{
    "Checklist": {
        "CS1_Results_vs_Conclusion": "PASS",
        "CS2_Plan_vs_Implementation": "FAIL"
    },
    "Rationale": {
        "CS1_Results_vs_Conclusion": "The documentation.pdf contains the ICLR 2024 paper with reported results. The demo notebook (fv_demo.ipynb) has no executed outputs saved (all 10 code cells have empty outputs). There are no saved result files (.json, .pt, .pkl) in the repository. Since there are no recorded results in the code implementation to compare against, there is no contradiction between the documentation conclusions and any recorded results.",
        "CS2_Plan_vs_Implementation": "The plan.md specifies 4 methodology steps. Steps 1-3 are implemented: (1) Causal mediation analysis in compute_indirect_effect.py, (2) FV testing across contexts in evaluate_function_vector.py, portability_eval.py, and natural_text_eval.

In [44]:
# Now we need to save the notebook to the evaluation directory
# The notebook is currently at /home/smallyan/eval_agent/notebooks/...
# We need to copy it to the repo's evaluation directory as self_matching.ipynb

import shutil

# Get the current notebook path
current_notebook = '/home/smallyan/eval_agent/notebooks/2026-01-14-17-37_CircuitAnalysisEval.ipynb'
target_path = f'{repo_path}/evaluation/self_matching.ipynb'

# Copy the notebook
shutil.copy(current_notebook, target_path)
print(f"✅ Copied notebook to: {target_path}")

# Verify the files exist
print("\n=== Evaluation Directory Contents ===")
for f in os.listdir(f'{repo_path}/evaluation'):
    print(f"  📄 {f}")